# CS-245 Machine Learning Project
## Phase 1: Data Preprocessing & Exploratory Data Analysis
### Dataset: Pakistan Property Listings

---

**Group Members:** 
Sukaina Nasir, 
Zayna Qasim, 
Hamna Shah  
**Course:** CS-245 — Machine Learning 

---

This notebook covers the complete **Phase 1** pipeline:
1. Loading & understanding the raw dataset
2. Cleaning and parsing messy columns (Price, Area, Purpose)
3. Handling missing values and outliers
4. Encoding categorical features
5. Exploratory Data Analysis (EDA) with 7 visualisations

> **Problem Type:** Regression — Predicting property prices (PKR) based on features like area, bedrooms, city, and property type.


---
## Step 0 — Install Dependencies & Import Libraries

We begin by installing any required libraries and importing all necessary packages for data manipulation, preprocessing, and visualisation.


In [ ]:
# Install libraries (only needed in Colab)
!pip install seaborn --quiet


In [ ]:
# ── Core Libraries ──────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# ── Scikit-learn ─────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")

# ── Plot Styling ─────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#f8f9fa",
    "axes.grid":        True,
    "grid.color":       "#e0e0e0",
    "grid.linestyle":   "--",
    "font.family":      "DejaVu Sans",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
})
PALETTE = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#44BBA4", "#E94F37"]

print(" All libraries imported successfully.")


---
## Step 1 — Load the Dataset

We load the raw CSV file containing **116,033 property listings** scraped from a Pakistani real-estate portal.

### Dataset Features

| Column | Description |
|--------|-------------|
| `type` | Property type (House, Flat, Plot, etc.) |
| `purpose` | Listing intent (For Sale / For Rent) |
| `area` | Property size as a string (e.g., "5 Marla", "1 Kanal") |
| `bedroom` | Number of bedrooms |
| `bath` | Number of bathrooms |
| `price` | Price as a string (e.g., "PKR 2.5 Crore") |
| `location` | Sub-locality name |
| `location_city` | City name |

> Several columns are stored as **raw strings** and require heavy cleaning before they can be used in ML models.


In [ ]:
# ── Load CSV ─────────────────────────────────────────────
# If running locally, update the path below.
# In Colab, upload the file first or mount Google Drive.

df = pd.read_csv("property_data.csv")   # <-- update path if needed

print(f"Shape      : {df.shape}")
print(f"Columns    : {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()


In [ ]:
# ── Basic Info ───────────────────────────────────────────
print("=== Dataset Info ===")
df.info()


In [ ]:
# ── Missing Values ───────────────────────────────────────
print("Missing values per column:")
df.isnull().sum()


---
## Step 2 — Parse & Clean the `price` Column

The price column contains values like:
- `"PKR\n2.65 Crore"`
- `"PKR\n40 Lakh"`
- `"PKR\n1.2 Arab"`

### Conversion Strategy
We convert everything to a **common unit: Pakistani Rupees in Lakhs** so all values are on the same numeric scale.

| Unit | Conversion |
|------|-----------|
| 1 Crore | = 100 Lakh |
| 1 Arab | = 10,000 Lakh |
| 1 Lakh | = 1 Lakh (base unit) |

This creates a new numeric column `price_lakh`.


In [ ]:
def parse_price(p):
    """
    Convert messy price strings to a float in Lakhs (PKR).
    Examples:
        'PKR\\n2.65 Crore'  →  265.0
        'PKR\\n40 Lakh'     →   40.0
        'PKR\\n1 Arab'      → 10000.0
    """
    try:
        p = str(p).replace("PKR", "").replace("\n", "").strip().replace(",", "")
        if "Crore" in p:
            return float(p.replace("Crore", "").strip()) * 100
        elif "Lakh" in p:
            return float(p.replace("Lakh", "").strip())
        elif "Arab" in p:
            return float(p.replace("Arab", "").strip()) * 10000
        else:
            return float(p)
    except:
        return np.nan

# Apply the parser
df["price_lakh"] = df["price"].apply(parse_price)

print(f"NaN after parsing : {df['price_lakh'].isna().sum()}")
print(f"Price range       : {df['price_lakh'].min():.1f} – {df['price_lakh'].max():.1f} Lakh")
print(f"\nSample parsed values:")
df[["price", "price_lakh"]].dropna().head(8)


---
## Step 3 — Parse & Clean the `area` Column

Area is stored as strings like `"1 Kanal"`, `"8 Marla"`, `"250 Sq. Yd."`.

### Conversion Strategy
We convert all area units to **Marla** — the most common unit in Pakistani real estate.

| Unit | Conversion |
|------|-----------|
| 1 Kanal | = 20 Marla |
| 1 Sq. Yard | = 1/25.29 Marla |
| 1 Sq. Ft. | = 1/272.25 Marla |

This creates a new numeric column `area_marla`.


In [ ]:
def parse_area(a):
    """
    Convert area strings to a float in Marla.
    Examples:
        '1 Kanal'     →  20.0
        '8 Marla'     →   8.0
        '250 Sq. Yd.' →   9.88
    """
    try:
        a = str(a).strip()
        if "Kanal" in a:
            return float(a.replace("Kanal", "").strip()) * 20
        elif "Marla" in a:
            return float(a.replace("Marla", "").strip())
        elif "Sq. Yd." in a:
            return float(a.replace("Sq. Yd.", "").strip()) / 25.2929
        elif "Sq. Ft." in a:
            return float(a.replace("Sq. Ft.", "").strip()) / 272.25
        else:
            return np.nan
    except:
        return np.nan

df["area_marla"] = df["area"].apply(parse_area)

print(f"NaN after parsing : {df['area_marla'].isna().sum()}")
print(f"Area range        : {df['area_marla'].min():.2f} – {df['area_marla'].max():.2f} Marla")
print(f"\nSample parsed values:")
df[["area", "area_marla"]].dropna().head(8)


---
## Step 4 — Standardise the `purpose` Column

The purpose column has **154 unique messy values** like:
- `"For Sale F-8"`, `"For Sale Owner"`, `"For Sale At DHA 2"` → all mean **For Sale**
- `"For Rent Portion"`, `"For Office Rent"` → all mean **For Rent**

We simplify this to just **3 clean labels**: `For Sale`, `For Rent`, `Other`.


In [ ]:
def clean_purpose(p):
    p = str(p).lower()
    if "rent" in p:
        return "For Rent"
    elif "sale" in p or "buy" in p or "luxury" in p:
        return "For Sale"
    else:
        return "Other"

df["purpose_clean"] = df["purpose"].apply(clean_purpose)

print("Purpose value counts after cleaning:")
print(df["purpose_clean"].value_counts())


---
## Step 5 — Clean `bedroom` and `bath` Columns

These columns contain non-numeric values like `"Studio"`, `"10+"`, and decimal strings like `"4.0"`. We convert them all to clean numeric values.

- `"Studio"` → `0` bedrooms
- `"10+"` → `10`
- `"4.0"` → `4`


In [ ]:
def clean_int_col(val):
    try:
        v = str(val).strip().replace("+", "").replace("Studio", "0")
        return float(v)
    except:
        return np.nan

df["bedroom"] = df["bedroom"].apply(clean_int_col)
df["bath"]    = df["bath"].apply(clean_int_col)

# Also clean the city column (strip leading/trailing whitespace)
df["city"] = df["location_city"].str.strip()

print("Bedroom sample values:", df["bedroom"].dropna().unique()[:10])
print("Bath sample values   :", df["bath"].dropna().unique()[:10])


---
## Step 6 — Handle Missing Values & Remove Outliers

### Missing Values
We drop rows where the **target (`price_lakh`)** or key feature (`area_marla`) is NaN, since these rows cannot be used in modelling.

### Outlier Removal
We use the **IQR (Interquartile Range)** method with a **3× multiplier** (conservative — keeps more data than the standard 1.5×):

$$\text{Keep if: } Q_1 - 3 \times IQR \leq x \leq Q_3 + 3 \times IQR$$

This removes extreme values (e.g., a property listed at 9,000,000 Lakhs) while preserving the natural spread.


In [ ]:
# ── Drop rows with missing critical values ────────────────
df_clean = df.dropna(subset=["price_lakh", "area_marla"]).copy()
print(f"Rows before dropping NaN : {len(df)}")
print(f"Rows after               : {len(df_clean)}")

# ── IQR Outlier Removal ───────────────────────────────────
def remove_outliers_iqr(df, col, factor=3.0):
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    return df[(df[col] >= Q1 - factor * IQR) & (df[col] <= Q3 + factor * IQR)]

before = len(df_clean)
df_clean = remove_outliers_iqr(df_clean, "price_lakh")
df_clean = remove_outliers_iqr(df_clean, "area_marla")

print(f"\nRows removed by IQR filter : {before - len(df_clean)}")
print(f"Dataset size after cleaning : {len(df_clean)}")


---
## Step 7 — Filter to Relevant Subsets

To build a clean and focused model, we:
1. Keep only `For Sale` and `For Rent` listings (drop the "Other" 14 rows)
2. Keep only the **top 7 property types** by frequency
3. Keep only **cities with ≥ 200 listings** to ensure enough training data per city


In [ ]:
# Keep main purposes only
df_clean = df_clean[df_clean["purpose_clean"].isin(["For Sale", "For Rent"])]

# Keep top 7 property types
top_types = df_clean["type"].value_counts().head(7).index
df_clean  = df_clean[df_clean["type"].isin(top_types)]

# Keep cities with sufficient data
city_counts = df_clean["city"].value_counts()
top_cities  = city_counts[city_counts >= 200].index
df_clean    = df_clean[df_clean["city"].isin(top_cities)]

print(f"Final shape after filtering : {df_clean.shape}")
print(f"\nProperty types kept:")
print(df_clean["type"].value_counts())
print(f"\nCities kept: {len(top_cities)}")


---
## Step 8 — Encode Categorical Features

Machine learning models require **numeric inputs**. We use **Label Encoding** to convert text categories into integers.

| Column | Example | Encoded |
|--------|---------|---------|
| `type` | "House" | 2 |
| `purpose_clean` | "For Sale" | 1 |
| `city` | "Lahore" | 5 |

> **Note:** For tree-based models (Random Forest), Label Encoding is appropriate. For linear models, One-Hot Encoding would be better — we handle this in Phase 2.


In [ ]:
le_type    = LabelEncoder()
le_purpose = LabelEncoder()
le_city    = LabelEncoder()

df_clean["type_enc"]    = le_type.fit_transform(df_clean["type"])
df_clean["purpose_enc"] = le_purpose.fit_transform(df_clean["purpose_clean"])
df_clean["city_enc"]    = le_city.fit_transform(df_clean["city"])

print("Label encoding mapping — Property Type:")
for i, cls in enumerate(le_type.classes_):
    print(f"  {i} → {cls}")


---
## Step 9 — Build the Feature Matrix

We select the final features and target, drop any remaining NaNs, and apply **StandardScaler** to normalise the feature values.

**StandardScaler** transforms each feature to have:
- Mean = 0
- Standard deviation = 1

This is especially important for Linear Regression (Phase 2), which is sensitive to feature scale.


In [ ]:
FEATURES = ["area_marla", "bedroom", "bath", "type_enc", "purpose_enc", "city_enc"]
TARGET   = "price_lakh"

df_model = df_clean[FEATURES + [TARGET]].dropna().copy()

print(f"Model-ready dataset : {df_model.shape}")
print(f"\nFeature summary:")
df_model[FEATURES].describe().round(2)


In [ ]:
# ── Scale features ───────────────────────────────────────
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(df_model[FEATURES])
X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES, index=df_model.index)

print("Scaled feature stats (mean ≈ 0, std ≈ 1):")
X_scaled_df.describe().round(3)


---
## Step 10 — Save Processed Data

We save the cleaned and encoded dataset to a CSV file. This will be loaded directly in Phase 2 (Model Development) and Phase 3 (Clustering) — no need to repeat preprocessing.


In [ ]:
df_model.to_csv("processed_data.csv", index=False)
print("Processed data saved → processed_data.csv")
print(f"   Shape : {df_model.shape}")
print(f"   Target range : {df_model[TARGET].min():.1f} – {df_model[TARGET].max():.1f} Lakhs")


---
## Step 11 — Exploratory Data Analysis (EDA)

EDA helps us understand the data **before** modelling. We generate 7 plots covering:

1. Price distribution (raw + log-transformed)
2. Area vs Price scatter
3. Median price by city
4. Price distribution by property type
5. Feature correlation heatmap
6. Median price by number of bedrooms
7. For Sale vs For Rent breakdown

> **Why log-transform price?** The raw price distribution is heavily right-skewed (a few very expensive properties). Log transformation makes it more symmetric, which helps linear models.


In [ ]:
# ── Plot 1: Price Distribution ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Price Distribution (PKR in Lakhs)", fontsize=14, fontweight="bold")

axes[0].hist(df_model["price_lakh"], bins=60, color=PALETTE[0], edgecolor="white", alpha=0.85)
axes[0].set_title("Raw Price Distribution")
axes[0].set_xlabel("Price (Lakhs)")
axes[0].set_ylabel("Frequency")

axes[1].hist(np.log1p(df_model["price_lakh"]), bins=60, color=PALETTE[1], edgecolor="white", alpha=0.85)
axes[1].set_title("Log-Transformed Price Distribution")
axes[1].set_xlabel("log(1 + Price)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("plot1_price_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("📌 Observation: Raw prices are right-skewed. Log-transform reveals a near-normal distribution, which is better for linear models.")


In [ ]:
# ── Plot 2: Area vs Price Scatter ────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
sample = df_model.sample(min(5000, len(df_model)), random_state=42)
scatter = ax.scatter(
    sample["area_marla"], sample["price_lakh"],
    c=sample["purpose_enc"], cmap="coolwarm",
    alpha=0.4, s=15, edgecolors="none"
)
ax.set_xlabel("Area (Marla)")
ax.set_ylabel("Price (Lakhs)")
ax.set_title("Area vs Price  (colour = Purpose: blue=Rent, red=Sale)")
plt.colorbar(scatter, ax=ax, label="Purpose Encoded")
plt.tight_layout()
plt.savefig("plot2_area_vs_price.png", dpi=150, bbox_inches="tight")
plt.show()
print("Observation: Clear positive correlation between area and price. 'For Sale' properties dominate higher price ranges.")


In [ ]:
# ── Plot 3: Median Price by City ─────────────────────────
city_price = (
    df_model.join(df_clean["city"])
    .groupby("city")["price_lakh"]
    .median()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(city_price.index, city_price.values, color=PALETTE[0], edgecolor="white")
ax.set_xlabel("Median Price (Lakhs)")
ax.set_title("Median Property Price by City (Top 15)")
ax.invert_yaxis()
for bar, val in zip(bars, city_price.values):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, f"{val:.0f}L", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("plot3_price_by_city.png", dpi=150, bbox_inches="tight")
plt.show()
print("Observation: Islamabad and Lahore have significantly higher median property prices, confirming city is an important feature.")


In [ ]:
# ── Plot 4: Price by Property Type ───────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
type_order = (
    df_model.join(df_clean["type"])
    .groupby("type")["price_lakh"]
    .median()
    .sort_values(ascending=False)
    .index
)
sns.boxplot(
    data=df_model.join(df_clean["type"]),
    x="type", y="price_lakh",
    order=type_order, palette=PALETTE, ax=ax, fliersize=2
)
ax.set_title("Price Distribution by Property Type")
ax.set_xlabel("Property Type")
ax.set_ylabel("Price (Lakhs)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("plot4_price_by_type.png", dpi=150, bbox_inches="tight")
plt.show()
print("Observation: Farm Houses and Penthouses command premium prices. Plots show high variance due to varying locations.")


In [ ]:
# ── Plot 5: Feature Correlation Heatmap ──────────────────
fig, ax = plt.subplots(figsize=(8, 6))
corr = df_model.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="RdYlGn", center=0, ax=ax,
    linewidths=0.5, linecolor="white",
    annot_kws={"size": 10}
)
ax.set_title("Feature Correlation Heatmap", fontsize=13, pad=15)
plt.tight_layout()
plt.savefig("plot5_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Observation: area_marla has the strongest correlation with price_lakh. bedroom and bath are also positively correlated.")


In [ ]:
# ── Plot 6: Bedrooms vs Price ─────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
bed_price = df_model[df_model["bedroom"] <= 10].groupby("bedroom")["price_lakh"].median()
bars = ax.bar(bed_price.index, bed_price.values, color=PALETTE[2], edgecolor="white")
ax.set_xlabel("Number of Bedrooms")
ax.set_ylabel("Median Price (Lakhs)")
ax.set_title("Median Price by Number of Bedrooms")
for x, y in zip(bed_price.index, bed_price.values):
    ax.text(x, y + 1, f"{y:.0f}L", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("plot6_bedrooms_vs_price.png", dpi=150, bbox_inches="tight")
plt.show()
print("Observation: More bedrooms generally means higher price, with diminishing returns beyond 6 bedrooms.")


In [ ]:
# ── Plot 7: For Sale vs For Rent ─────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
purpose_counts = df_clean["purpose_clean"].value_counts()
ax.pie(
    purpose_counts.values,
    labels=purpose_counts.index,
    autopct="%1.1f%%",
    colors=[PALETTE[0], PALETTE[1]],
    startangle=140,
    wedgeprops={"edgecolor": "white", "linewidth": 2}
)
ax.set_title("Property Listing Purpose Breakdown", fontsize=13, pad=15)
plt.tight_layout()
plt.savefig("plot7_purpose_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(" Observation: ~75% of listings are For Sale. This imbalance will be accounted for in model training.")


---
## Phase 1 Summary

| Step | Action | Result |
|------|--------|--------|
| Price Parsing | Converted PKR strings → numeric Lakhs | |
| Area Parsing | Converted Kanal/Marla/SqFt → Marla | |
| Purpose Cleaning | 154 messy values → Sale/Rent/Other | |
| Bedroom/Bath Cleaning | Removed Studio, 10+, decimals | |
| Missing Values | Dropped rows with NaN price or area | |
| Outlier Removal | IQR 3× filter on price & area | |
| Encoding | Label encoding for type, purpose, city | |
| EDA | 7 visualisation plots generated | |

###  Final Dataset
- **38,350 clean rows** ready for modelling
- **6 features:** `area_marla`, `bedroom`, `bath`, `type_enc`, `purpose_enc`, `city_enc`
- **Target:** `price_lakh` (property price in Pakistani Rupees × Lakh)

---
>  **Next: Phase 2 — Supervised Model Development** (Linear Regression + Random Forest + XGBoost)
